# Universe D pair diagnosis

Full frozen-stack scorecard for Universe D share-class pairs (`WSO|WSO.B`, `HEI|HEI.A`, `NWS|NWSA`).
Uses `run_s2_backtest` + `US_ALPACA_D_REALISTIC` costs. Does **not** change STARs.

**Helpers:** `04_backtest/s2_coint/diagnosis.py` (`stack_scorecard`, `pair_deployment_table`, `plotly_pair_diagnosis`)

In [1]:
from __future__ import annotations

import os
import sys

import pandas as pd

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from backtest.s2_coint.diagnosis import (
    check_fill_timing,
    pair_deployment_table,
    plotly_pair_diagnosis,
    print_extreme_trades,
    stack_scorecard,
)
from backtest.s2_coint.research import (
    DEFAULT_STAR_STACK,
    frozen_pairs_for_universe,
    load_s1_weekly,
    load_star_stack,
    load_universe_panels,
    research_is_end_for,
)

USE_OOS = False
COMPARE_BASELINE_COSTS = True

stack = load_star_stack(DEFAULT_STAR_STACK)
universe = str(stack["UNIVERSE_STAR"])
bar = str(stack.get("BAR_STAR") or "1d")
pair_ids = frozen_pairs_for_universe(universe, bar=bar)
is_end = research_is_end_for(universe)
train, full = load_universe_panels(universe, bar, pair_ids)
panel = full if USE_OOS else train
s1_weekly = load_s1_weekly()

realistic = stack_scorecard(
    panel,
    stack,
    use_oos=USE_OOS,
    is_end=is_end,
    s1_weekly=s1_weekly,
    cost_profile="US_ALPACA_D_REALISTIC",
)
print("=== Realistic D costs ===")
print(realistic["metrics"])
display(pair_deployment_table(realistic["pair_trades"]))

if COMPARE_BASELINE_COSTS:
    baseline = stack_scorecard(
        panel,
        stack,
        use_oos=USE_OOS,
        is_end=is_end,
        s1_weekly=s1_weekly,
        cost_profile="US_ALPACA",
    )
    print("\n=== Baseline US_ALPACA (no borrow) ===")
    print(baseline["metrics"])

print_extreme_trades(realistic["pair_trades"], n=3)
timing = check_fill_timing(realistic["pair_trades"], panel)
display(timing)

for pid in pair_ids:
    g = panel.loc[panel["pair_id"] == pid]
    t = realistic["pair_trades"].loc[realistic["pair_trades"]["pair_id"] == pid]
    fig = plotly_pair_diagnosis(
        g,
        t,
        entry_z=float(realistic["config"].entry_z),
        z_window=int(realistic["config"].z_window),
        pair_returns=realistic["pair_returns"].get(pid),
        is_end=is_end,
        title=f"{pid} (full stack, realistic costs)",
    )
    fig.show()

=== Realistic D costs ===
{'ann_sharpe': 1.3753857608669167, 'max_drawdown': -0.09386941995823361, 'n_days': 8064, 'psr': 1.0, 'dsr_local': nan, 'dsr_stack': nan, 'skew': 1.3386848748238194, 'excess_kurtosis': 17.670975541305037, 'n_trials_local': None, 'n_trials_stack': None, 'corr_to_s1': -0.14743107283480394}


,pair_id,n_round_trips,median_hold_bars,median_entry_cost_bps,median_exit_cost_bps,median_pnl_pct
0,HEI|HEI.A,50,10.5,11.4,11.4,1.601440
1,NWS|NWSA,36,7.0,11.4,11.4,0.311591
2,WSO.B|WSO,484,3.0,11.4,11.4,1.540127



=== Baseline US_ALPACA (no borrow) ===
{'ann_sharpe': 1.4941960703571942, 'max_drawdown': -0.08725458184840085, 'n_days': 8064, 'psr': 1.0, 'dsr_local': nan, 'dsr_stack': nan, 'skew': 1.3686144019858606, 'excess_kurtosis': 17.704616204803344, 'n_trials_local': None, 'n_trials_stack': None, 'corr_to_s1': -0.1472353334923041}
=== Best 3 trades by net pnl_pct ===
  pair_id side_label entry_date  exit_date   pnl_pct  z_entry
WSO.B|WSO      short 2000-02-09 2000-02-25 28.884492 3.386193
HEI|HEI.A      short 2008-10-13 2008-10-16 20.728351 4.506574
WSO.B|WSO      short 2008-10-08 2008-10-13 15.106136 4.499739

=== Worst 3 trades by net pnl_pct ===
  pair_id side_label entry_date  exit_date    pnl_pct   z_entry
WSO.B|WSO      short 1991-08-08 1991-10-08 -18.613954  1.806495
WSO.B|WSO       long 2020-03-16 2020-03-17 -10.908627 -1.889910
WSO.B|WSO       long 2008-11-24 2008-12-02  -9.974448 -2.376768


,pair_id,entry_date,signal_date,ok_signal_before_fill,ok_fill_has_open,ok_not_same_bar_close_fill,ok_z_finite_on_signal,all_ok
0,HEI|HEI.A,1999-08-19,1999-08-18,True,True,True,True,True
1,HEI|HEI.A,1999-08-25,1999-08-24,True,True,True,True,True
2,HEI|HEI.A,1999-10-26,1999-10-25,True,True,True,True,True
3,HEI|HEI.A,2000-02-25,2000-02-24,True,True,True,True,True
4,HEI|HEI.A,2000-04-05,2000-04-04,True,True,True,True,True
...,...,...,...,...,...,...,...,...
565,WSO.B|WSO,2021-08-12,2021-08-11,True,True,True,True,True
566,WSO.B|WSO,2021-10-21,2021-10-20,True,True,True,True,True
567,WSO.B|WSO,2021-10-27,2021-10-26,True,True,True,True,True
568,WSO.B|WSO,2021-12-01,2021-11-30,True,True,True,True,True
